# Loading in our files:

In [3]:
import pandas as pd
import numpy as np
print(pd.__version__)
print(np.__version__)

3.0.0
2.4.2


In [4]:
from pathlib import Path

import pandas as pd
from tqdm.notebook import tqdm


In [5]:
files = sorted(Path("processed_chunks").glob("chunk_*.pkl"))

for file in files[:3]:
    df = pd.read_pickle(file)
    
    print(file.name)
    print(df.head())

# Loading and splitting data:

In [6]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

fake_categories = {"fake", "hate", "unreliable", "clickbait", "conspiracy", "junksci", "rumor"}
valid_types = fake_categories | {"reliable", "political", "bias"}

def load_chunks_logistic(files):
    dfs = []
    for file in tqdm(files):
        df = pd.read_parquet(file)
        df = df[df["type"].isin(valid_types)][["type", "tokens_stemmed", "domain"]]
        dfs.append(df)
    data = pd.concat(dfs, ignore_index=True)
    del dfs
    data["text"] = data["tokens_stemmed"].apply(lambda tokens: " ".join(tokens))
    data["text_and_domain"] = data["tokens_stemmed"].apply(lambda tokens: " ".join(tokens)) + " " + data["domain"]
    data["label"] = data["type"].apply(lambda x: 1 if x in fake_categories else 0)
    return data[["text", "text_and_domain", "label"]]

files = sorted(Path("processed_chunks_paquet").glob("chunk_*.parquet"))
all_data = load_chunks_logistic(files[:100])

# First split off 20% for val+test
train_data, temp_data = train_test_split(all_data, test_size=0.2, random_state=42)

# Split the 20% evenly into val and test
val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42)

X_train, y_train = train_data["text"], train_data["label"]
X_val,   y_val   = val_data["text"],   val_data["label"]
X_test,  y_test  = test_data["text"],  test_data["label"]

X_tr_domain = train_data["text_and_domain"]
X_v_domain  = val_data["text_and_domain"]
X_te_domain = test_data["text_and_domain"]

print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")



  0%|          | 0/100 [00:00<?, ?it/s]

Train: 712000, Val: 89000, Test: 89001


# Simple logistic regression

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import CountVectorizer #Doesnt do any normalization

# Vectorize
vectorizer = CountVectorizer(max_features=10000)
X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec   = vectorizer.transform(X_val)
X_test_vec  = vectorizer.transform(X_test)

#sm = SMOTE()  
#X_train_re, y_train_re = sm.fit_resample(X_train_vec, y_train)

# Train
model = LogisticRegression(
  max_iter=1000,
)
model.fit(X_train_vec, y_train)

# Evaluate on validation set
y_val_pred = model.predict(X_val_vec)
print("Validation:")
print(classification_report(y_val, y_val_pred, target_names=["reliable", "fake"]))

Validation:
              precision    recall  f1-score   support

    reliable       0.86      0.92      0.89     54640
        fake       0.86      0.76      0.81     34360

    accuracy                           0.86     89000
   macro avg       0.86      0.84      0.85     89000
weighted avg       0.86      0.86      0.86     89000



In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import CountVectorizer #Doesnt do any normalization

# Vectorize
vectorizer = CountVectorizer(max_features=10000)
X_train_vec = vectorizer.fit_transform(X_tr_domain)
X_val_vec   = vectorizer.transform(X_v_domain)
X_test_vec  = vectorizer.transform(X_te_domain)

# Train
model_two = LogisticRegression(max_iter=1000)
model_two.fit(X_train_vec, y_train)

# Evaluate on validation set
y_val_pred_two = model_two.predict(X_val_vec)
print("Validation:")
print(classification_report(y_val, y_val_pred_two, target_names=["reliable", "fake"]))

Validation:
              precision    recall  f1-score   support

    reliable       0.98      0.98      0.98     54640
        fake       0.97      0.96      0.97     34360

    accuracy                           0.97     89000
   macro avg       0.97      0.97      0.97     89000
weighted avg       0.97      0.97      0.97     89000



In [12]:
df_liar = pd.read_csv("liar_dataset/train.tsv", sep="\t", header=None)
df_liar.columns = ["id", "label", "statement", "subject", "speaker", 
              "job", "state", "party", "barely_true", "false_count",
              "half_true", "mostly_false", "pants_fire", "context"]

fake_categories_liar = {"false", "pants-on-fire", "barely-true"}
valid_types_liar     = fake_categories_liar | {"true", "mostly-true"}

def map_to_binary(label):
    if label in fake_categories_liar:
        return 1  # fake
    elif label in valid_types_liar:
        return 0  # real
    else:
        return None

liar_test = vectorizer.transform(df_liar["statement"])

# Apply binary mapping and drop unmapped rows (e.g. "half-true")
df_liar["binary_label"] = df_liar["label"].apply(map_to_binary)
df_liar = df_liar.dropna(subset=["binary_label"])
df_liar["binary_label"] = df_liar["binary_label"].astype(int)

liar_vec = vectorizer.transform(df_liar["statement"])
y_liar   = df_liar["binary_label"]

print("=== LIAR Dataset ===")
print(classification_report(y_liar, model.predict(liar_vec)))


=== LIAR Dataset ===
              precision    recall  f1-score   support

           0       0.50      0.87      0.63      3638
           1       0.50      0.13      0.21      3649

    accuracy                           0.50      7287
   macro avg       0.50      0.50      0.42      7287
weighted avg       0.50      0.50      0.42      7287

